# Training entrypoint (Google Colab)

Runs one full training experiment for the config in `cnn/config.py`: loads a
processed dataset, builds/fine-tunes a classifier (optionally in two phases —
head-only warmup then full fine-tune, per `config.FREEZE`), evaluates on
val/test, and saves all run artifacts (weights, metrics, history,
misclassified images) under `RUNS_ROOT`.

**Cell 1 (below)** is Colab-specific bootstrap: mounts Google Drive, clones
this repo, and copies the dataset from Drive to local Colab disk for faster
I/O. Adjust `DATA_ROOT`/`RUNS_ROOT`/the `git clone` URL for your own setup —
these paths are specific to the original author's Drive layout.

**Cell 2** is the actual training pipeline and does not depend on Colab; it
could be run locally by setting `DATA_ROOT`/`RUNS_ROOT` to local paths instead.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from getpass import getpass
token = getpass("GitHub token: ")

!rm -rf IIB-Project
!git clone https://{token}@github.com/aharris64/IIB-Project.git

%cd /content/IIB-Project/cnn

DATA_ROOT = "/content/drive/MyDrive/IIB_Project/data"
RUNS_ROOT = "/content/drive/MyDrive/IIB_Project/runs"

# Copy dataset to local SSD
import os, subprocess

dst = "/content/data/disc_centred_r4.0_cl34_augmented_lowres112"
src = "/content/drive/MyDrive/IIB_Project/data/disc_centred_r4.0_cl34_augmented_lowres112/"

os.makedirs(dst, exist_ok=True)
result = subprocess.run(
    ["rsync", "-avh", "--progress", "--timeout=60", src, dst],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    raise RuntimeError(f"rsync failed:\n{result.stderr}")

DATA_ROOT = "/content/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GitHub token: ··········
Cloning into 'IIB-Project'...
remote: Enumerating objects: 898, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (242/242), done.
remote: Total 898 (delta 46), reused 244 (delta 27), pack-reused 627 (from 2)
Receiving objects: 100% (898/898), 142.07 MiB | 21.09 MiB/s, done.
Resolving deltas: 100% (369/369), done.
/content/IIB-Project/cnn
sending incremental file list

sent 270.24K bytes  received 44 bytes  180.19K bytes/sec
total size is 212.27M  speedup is 785.36



## Training pipeline

Load data → build model → (two-phase or single-phase) train → evaluate best
checkpoint on val/test → save everything to `RUNS_ROOT/<model>_<timestamp>/`.

In [2]:
import torch
import numpy as np
import torch.nn as nn
from pathlib import Path
import json
import random
from datetime import datetime
from sklearn.metrics import f1_score, balanced_accuracy_score, confusion_matrix, classification_report
import csv

from train import train
from evaluate import evaluate
from models import build_model, get_backbone
from load_data import get_dataloaders
import config


def seed_everything(seed):
    """Seed random/numpy/torch (CPU + all CUDA devices) for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def trainable_parameters(model):
    """Return the list of parameters with requires_grad=True (for the optimizer)."""
    return [p for p in model.parameters() if p.requires_grad]


def unfreeze(module):
    # Mirrors models.unfreeze; kept local so this notebook stays self-contained.
    for p in module.parameters():
        p.requires_grad = True


def save_json(path: Path, obj) -> None:
    """Write obj as indented JSON to path, creating parent directories as needed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(obj, f, indent=2)


def get_classifier(model, model_name):
    """Return the classification head module for a given model."""
    if model_name in ("mobilenet_v3", "mobilenet_v3_small", "mobilenet_v2", "efficientnet_b0",
                      "efficientnet_lite0", "efficientnet_lite1"):
        return model.classifier
    elif model_name == "resnet":
        return model.fc
    elif model_name == "squeezenet":
        return model.classifier
    elif model_name == "ghostnet":
        return model.classifier
    else:
        raise ValueError(f"Unknown model_name='{model_name}'")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(config.SEED)

# Load data
train_loader, val_loader, test_loader = get_dataloaders(
    root_folder=DATA_ROOT,
    dataset=config.DATASET,
    batch_size=config.BATCH_SIZE,
)
train_ds = train_loader.dataset
print("Dataset: ", config.DATASET)

# Build model — always start with head frozen for phase 1
model = build_model(config.MODEL_NAME, config.NUM_CLASSES, freeze="head")
model = model.to(device)
print("Model: ", config.MODEL_NAME)
print("Freeze mode: ", config.FREEZE)

# Class weights — inverse-frequency, to counter the dataset's class imbalance
counts = np.bincount([y for _, y in train_ds.samples])
weights = counts.sum() / (len(counts) * counts)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
print("Train class counts:", counts.tolist())
print("Class weights:", class_weights.tolist())

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Save experiment metadata
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path(RUNS_ROOT) / f"{config.MODEL_NAME}_{run_id}"
out_dir.mkdir(parents=True, exist_ok=True)

cfg_snapshot = {k: getattr(config, k) for k in dir(config) if k.isupper()}
save_json(out_dir / "config.json", cfg_snapshot)

run_meta = {
    "run_id": run_id,
    "timestamp": datetime.now().isoformat(),
    "device": str(device),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
    "seed": int(config.SEED),
    "train_class_counts": counts.tolist(),
    "class_weights": class_weights.detach().cpu().tolist(),
}
save_json(out_dir / "run_meta.json", run_meta)

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

print("Starting training...")
batch = next(iter(train_loader))
print(f"DataLoader OK: {batch[0].shape}")

if config.FREEZE == "two_phase":
    # ------------------------------------------------------------------
    # Phase 1: train head only for PHASE1_EPOCHS epochs
    # ------------------------------------------------------------------
    print(f"\n--- Phase 1: head only ({config.PHASE1_EPOCHS} epochs) ---")
    optimizer_p1 = torch.optim.AdamW(
        trainable_parameters(model),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
    )
    _, _, history_p1 = train(
        model,
        train_loader,
        val_loader,
        optimizer_p1,
        criterion,
        device,
        num_epochs=config.PHASE1_EPOCHS,
        patience=config.PATIENCE,
    )

    # ------------------------------------------------------------------
    # Phase 2: unfreeze backbone, fine-tune end-to-end with lower LR
    # ------------------------------------------------------------------
    print(f"\n--- Phase 2: full fine-tune ---")
    backbone = get_backbone(model, config.MODEL_NAME)
    unfreeze(backbone)

    # Backbone fine-tunes at 10% of the head's LR — already ImageNet-pretrained,
    # so a smaller step size avoids destroying those features.
    optimizer_p2 = torch.optim.AdamW([
        {"params": backbone.parameters(),                          "lr": config.LEARNING_RATE * 0.1},
        {"params": get_classifier(model, config.MODEL_NAME).parameters(), "lr": config.LEARNING_RATE},
    ], weight_decay=config.WEIGHT_DECAY)

    best_epoch, best_state, history_p2 = train(
        model,
        train_loader,
        val_loader,
        optimizer_p2,
        criterion,
        device,
        num_epochs=config.NUM_EPOCHS,
        patience=config.PATIENCE,
    )

    # Merge histories so the full curve is saved
    history = {
        "phase1": history_p1,
        "phase2": history_p2,
    }

else:
    # Single-phase training (freeze="none" or freeze="head")
    optimizer = torch.optim.AdamW(
        trainable_parameters(model),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
    )
    best_epoch, best_state, history = train(
        model,
        train_loader,
        val_loader,
        optimizer,
        criterion,
        device,
        config.NUM_EPOCHS,
        config.PATIENCE,
    )

# ---------------------------------------------------------------------------
# Evaluate best model
# ---------------------------------------------------------------------------

model.load_state_dict(best_state)
model.to(device)

val_loss,  y_true_v, y_pred_v, y_prob_v = evaluate(model, val_loader,  device, criterion)
test_loss, y_true_t, y_pred_t, y_prob_t = evaluate(model, test_loader, device, criterion)

# Save history
save_json(out_dir / "history.json", history)

# Metrics
results = {
    "best_epoch": int(best_epoch),
    "val": {
        "loss": float(val_loss),
        "macro_f1": float(f1_score(y_true_v, y_pred_v, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(y_true_v, y_pred_v)),
        "confusion_matrix": confusion_matrix(y_true_v, y_pred_v).tolist(),
        "classification_report": classification_report(y_true_v, y_pred_v, digits=4),
    },
    "test": {
        "loss": float(test_loss),
        "macro_f1": float(f1_score(y_true_t, y_pred_t, average="macro")),
        "balanced_acc": float(balanced_accuracy_score(y_true_t, y_pred_t)),
        "confusion_matrix": confusion_matrix(y_true_t, y_pred_t).tolist(),
        "classification_report": classification_report(y_true_t, y_pred_t, digits=4),
    },
}
save_json(out_dir / "metrics.json", results)

# Save predictions
np.savez_compressed(out_dir / "predictions_val.npz",
                    y_true=y_true_v, y_pred=y_pred_v, y_prob=y_prob_v)
np.savez_compressed(out_dir / "predictions_test.npz",
                    y_true=y_true_t, y_pred=y_pred_t, y_prob=y_prob_t)

# Save best weights
torch.save(best_state, out_dir / "best_model_state_dict.pt")

print(f"Saved run artifacts to: {out_dir}")


# ---------------------------------------------------------------------------
# Save misclassified images
# ---------------------------------------------------------------------------

def save_misclassified_csv(out_path: Path, dataset, y_true, y_pred):
    """Write filename/y_true/y_pred rows for every misclassified sample in dataset."""
    rows = []
    for (fp, _), yt, yp in zip(dataset.samples, y_true, y_pred):
        if int(yt) != int(yp):
            rows.append([Path(fp).name, int(yt), int(yp)])
    with out_path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["filename", "y_true", "y_pred"])
        w.writerows(rows)


save_misclassified_csv(out_dir / "misclassified_val.csv",  val_loader.dataset,  y_true_v, y_pred_v)
save_misclassified_csv(out_dir / "misclassified_test.csv", test_loader.dataset, y_true_t, y_pred_t)

Class mapping: {'normal': 0, 'papilledema': 1, 'pseudopapilledema': 2}
Train size: 11574 Val size: 410 Test size: 421
Dataset:  disc_centred_r4.0_cl34_augmented_lowres14
Model:  mobilenet_v3_small
Freeze mode:  two_phase
Train class counts: [8940, 1374, 1260]
Class weights: [0.43154361844062805, 2.8078603744506836, 3.0619046688079834]
Starting training...
DataLoader OK: torch.Size([32, 3, 224, 224])

--- Phase 1: head only (10 epochs) ---
[Epoch 1] Train loss: 0.8283 | Val loss: 0.6515 | Val macro-F1: 0.4805
[Epoch 2] Train loss: 0.6064 | Val loss: 0.5347 | Val macro-F1: 0.7445
[Epoch 3] Train loss: 0.5217 | Val loss: 0.4933 | Val macro-F1: 0.7454
[Epoch 4] Train loss: 0.4754 | Val loss: 0.4507 | Val macro-F1: 0.7638
[Epoch 5] Train loss: 0.4517 | Val loss: 0.4174 | Val macro-F1: 0.7686
[Epoch 6] Train loss: 0.4255 | Val loss: 0.3902 | Val macro-F1: 0.7829
[Epoch 7] Train loss: 0.4177 | Val loss: 0.3732 | Val macro-F1: 0.7797
[Epoch 8] Train loss: 0.3956 | Val loss: 0.3580 | Val macro-